In [1]:
# Nädal: 7    Meeskond: Sales Analytics   
# ROLL A: Data Loading (Andmete laadimine)

# Import pandas libary

import pandas as pd

# Load sales and customers data from cvs files

df_sales = pd.read_csv('sales.csv')

print(' === Esimesed 5 sales tabeli rida === \n')
print(df_sales.head())
print('\n === Ridade ja veergude arv sales tabelis === \n')
print(df_sales.shape)

df_customers = pd.read_csv('customers.csv')

print('\n === Esimesed 5 customers tabeli rida === \n')
print(df_customers.head())
print('\n === Ridade ja veergude arv customers tabelis === \n')
print(df_customers.shape)

# Merge sales and customers dataframes on customer_id

df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

print('\n === Esimesed 5 ühendatud tabeli rida === \n')
print(df.head())
print('\n === Ridade ja veergude arv ühendatud tabelis === \n')
print(df.shape)
print('\n === Andmetüübid ühendatud tabelis === \n')
print(df.dtypes)

 === Esimesed 5 sales tabeli rida === 

   sale_id        invoice_id   sale_date  customer_id  product_id  quantity  \
0        1  INV-202301-00001  2023-01-10       2588.0        1274         2   
1        2  INV-202301-00002  2023-01-16       4338.0        1207         2   
2        3  INV-202301-00003  2023-01-05       4673.0        1264         1   
3        4  INV-202301-00004  2023-01-02       4677.0        1341         3   
4        5  INV-202301-00005  2023-01-13       2390.0        1284         1   

   unit_price  total_price channel store_location payment_method  
0      234.79       469.58    pood        Tallinn          kaart  
1      241.13       482.26    pood          Pärnu      järelmaks  
2      258.46       221.19    pood          Pärnu      järelmaks  
3       45.21       135.63    pood          Tartu       sularaha  
4       99.57        99.57    pood          Tartu          kaart  

 === Ridade ja veergude arv sales tabelis === 

(15234, 11)

 === Esimesed 5 custo

In [2]:
# ROLL B: Data Cleaning (Andmete puhastamine)

# Print initial df shape

print(' === Ühendatud tabeli read ja veerud enne puhastamist === \n')
print(df.shape)

# Find the number of duplicates in the invoice_id column

print('\n === Duplikaatide arv === \n')
print('Duplikaadid:', df.duplicated(subset=['invoice_id']).sum())

# Drop duplicates

df = df.drop_duplicates(subset=['invoice_id'], keep='first')

# Check NULL values

print('\n === NULL-väärtuste arv === \n')
print("NULL-id:\n", df.isnull().sum())

# Remove rows where there the columns customer_id, sale_date and total_price contain NULL values

df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Convert sale_date column to the correct data type

df['sale_date'] = df['sale_date'].apply(lambda x: pd.to_datetime(x, format='%d/%m/%Y')
                                        if '/' in x else pd.to_datetime(x)
                                        )

# Keep 26 months of data starting from January 2023

before = len(df) # Enne kuupäevavahemiku filtreerimist

df = df[(df['sale_date'] >= '2023-01-01') & (df['sale_date'] < '2025-03-01')]

print('\n === Kuupäevavahemiku tõttu eemaldatud read === \n') 
print(before - len(df))

# Check the outliers in the total_price column

print('\n === Negatiivsete total_price väärtuste arv === \n')
print((df['total_price'] < 0).sum())

# Remove rows where the total_price column contains negative values

df = df[df['total_price'] > 0]

# Cleaning report

print('\n === Puhastamisraport === \n')
print(f'Puhastatud tabel: {df.shape[0]} rida, {df['customer_id'].nunique()} klienti')
print(f'Kuupäevavahemik: {df['sale_date'].min().date()} - {df['sale_date'].max().date()}')

 === Ühendatud tabeli read ja veerud enne puhastamist === 

(15234, 19)

 === Duplikaatide arv === 

Duplikaadid: 5116

 === NULL-väärtuste arv === 

NULL-id:
 sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

 === Kuupäevavahemiku tõttu eemaldatud read === 

28

 === Negatiivsete total_price väärtuste arv === 

179

 === Puhastamisraport === 

Puhastatud tabel: 8923 rida, 2540 klienti
Kuupäevavahemik: 2023-01-01 - 2025-02-28


In [3]:
# ROLL C: Analysis — RFM kliendisegmenteerimine

# Define the reference date

today = pd.to_datetime('2025-02-28')

# Find the last sale date

last_sale_date = df.groupby('customer_id')['sale_date'].max()

# Calculate the recency in days for each customer

recency_days = (today - last_sale_date).dt.days.reset_index(name='recency_days')

# Find the number of sales for each customer

frequency = df.groupby('customer_id')['sale_id'].count().reset_index(name='frequency')

# Find each customer's total spending

monetary = df.groupby('customer_id')['total_price'].sum().reset_index(name='monetary')

# Merge recency, frequency and monetary into a single dataframe

rfm = pd.merge(recency_days, frequency, on='customer_id', how='inner')
rfm = pd.merge(rfm, monetary, on='customer_id', how='inner')

# Set scores for recency, frequency and monetary 

rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5])

# Calculate the RFM score

rfm['RFM_score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Create segments based on RFM score

rfm['segment'] = rfm['RFM_score'].apply(lambda x: 'VIP Champions' if 13 <= x <= 15
                                                else 'Loyal' if  10 <= x <= 12
                                                else 'Potential' if  7 <= x <= 9
                                                else 'At risk' if 4 <= x <= 6
                                                else 'Lost')

# Summary of the RFM segments, proportions in percentages

segment_summary = rfm['segment'].value_counts().reset_index()
segment_summary.columns = ['Segment', 'Klientide arv']
segment_summary['Osakaal %'] = (segment_summary['Klientide arv'] / segment_summary['Klientide arv'].sum() * 100).round(1)

print('\n === RFM segmentide kokkuvõte === \n')
print(segment_summary)


 === RFM segmentide kokkuvõte === 

         Segment  Klientide arv  Osakaal %
0      Potential            768       30.2
1          Loyal            678       26.7
2        At risk            524       20.6
3  VIP Champions            453       17.8
4           Lost            117        4.6
